# Old training data: quiet positions from the Lichess evaluation database (Kaggle, CPU only)

**Step 1 of 2** for the next network (step 2 is notebook 08b, the training). This notebook **only builds a dataset**, it does not train. Run it with **Save & Run All** on a CPU session (Accelerator: None, Internet on). It does not use GPU hours.

It streams the Lichess evaluation database (`lichess_db_eval.jsonl.zst`, more than 400 million positions with a Stockfish evaluation) and keeps the positions that

- were analysed to Stockfish depth >= `min_depth` (25)
- are quiet: the best move is not a capture or promotion

This is the same filter as in notebooks 05, 06 and 08: about 143 million positions. The label is the score for white as a number between -1 and 1 (`tanh` of the centipawns, mate = -1 or 1).

It takes a few hours. The download is never stored: it is decompressed and read on the fly, and the heavy part (reading the JSON lines) runs on all CPU cores at the same time.

**Output** (in `/kaggle/working/lichess_evals_cache`, about 5 GB):

- `packed.npy`: the boards, 32 bytes per position (two squares per byte, the same packing as `lichess_games.bin` of notebook 07)
- `side.npy`: 1 if white is to move, 0 if black is to move
- `labels.npy`: the label for white (float16)
- `done.json`: the number of positions and the depth filter

**Afterwards:** in notebook 08b use **Add Input** -> *Your Work* -> this notebook. Notebook 08b finds the cache under `/kaggle/input/`.

In [ ]:
!pip install -q zstandard

In [ ]:
import collections
import json
import math
import multiprocessing
import os
import time
import urllib.request

import numpy as np
import zstandard

## Settings

In [ ]:
notebook_start = time.time()

max_lichess_positions = 150_000_000
max_load_minutes = 600
min_depth = 25

url = "https://database.lichess.org/lichess_db_eval.jsonl.zst"
out_dir = "/kaggle/working"
cache_dir = os.path.join(out_dir, "lichess_evals_cache")
log_file = os.path.join(out_dir, "lichess_evals.log")
num_workers = os.cpu_count()

print("CPU cores:", num_workers)

## Helpers

In [ ]:
def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")

## From a JSON line to a training position

A line of the database looks like this (the score is always for white):

`{"fen": "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq -", "evals": [{"pvs": [{"cp": 18, "line": "c7c5 g1f3 ..."}], "depth": 36}, ...]}`

Only the deepest evaluation is used. The first move of its best line (`pv`) decides if the position is quiet: no promotion, no piece on the target square and no pawn moving to another file (en passant).

The board is stored as 64 squares (a1 = 0, h8 = 63) with a code per square (0 empty, 1-6 white pawn to king, 7-12 black pawn to king), two squares per byte.

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"
CACHE_NAMES = ["packed", "side", "labels"]


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    index = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            index += int(char)
        else:
            row[index ^ 56] = PIECE_CODES[char]
            index += 1
    return row, color == "w"


def square_index(square):
    return (int(square[1]) - 1) * 8 + (ord(square[0]) - ord("a"))


def is_quiet(row, move):
    if len(move) == 5:
        return False
    from_square, to_square = square_index(move[:2]), square_index(move[2:4])
    if row[to_square] != 0:
        return False
    if row[from_square] in (1, 7) and move[0] != move[2]:
        return False
    return True


def score_to_label(cp, mate):
    if mate is not None:
        return 1.0 if mate > 0 else -1.0
    return 2 / (1 + math.exp(-0.00368208 * cp)) - 1


def parse_chunk(chunk):
    rows, sides, values = [], [], []
    num_lines = 0
    for line in chunk.splitlines():
        num_lines += 1
        data = json.loads(line)
        if not data["evals"]:
            continue
        deepest = max(data["evals"], key=lambda e: e["depth"])
        pv = deepest["pvs"][0]
        if deepest["depth"] < min_depth or not pv.get("line"):
            continue
        row, white = parse_fen(data["fen"])
        if not is_quiet(row, pv["line"].split()[0]):
            continue
        rows.append(row[0::2] | (row[1::2] << 4))
        sides.append(white)
        values.append(score_to_label(pv.get("cp"), pv.get("mate")))
    packed = np.array(rows, dtype=np.uint8).reshape(-1, 32)
    return packed, np.array(sides, dtype=np.uint8), np.array(values, dtype=np.float16), num_lines


def read_chunks(reader, chunk_size=16_000_000):
    rest = b""
    while True:
        block = reader.read(chunk_size)
        if not block:
            break
        block = rest + block
        end = block.rfind(b"\n") + 1
        rest = block[end:]
        yield block[:end]
    if rest:
        yield rest


def load_positions(max_positions, max_minutes):
    packed = np.zeros((max_positions, 32), dtype=np.uint8)
    side = np.zeros(max_positions, dtype=np.uint8)
    labels = np.zeros(max_positions, dtype=np.float16)
    count, num_lines, next_log = 0, 0, 5_000_000
    start = time.time()

    with urllib.request.urlopen(url) as response, multiprocessing.get_context("fork").Pool(num_workers) as pool:
        chunks = read_chunks(zstandard.ZstdDecompressor().stream_reader(response))
        pending = collections.deque()
        while count < max_positions and time.time() - start < max_minutes * 60:
            while len(pending) < 2 * num_workers:
                chunk = next(chunks, None)
                if chunk is None:
                    break
                pending.append(pool.apply_async(parse_chunk, (chunk,)))
            if not pending:
                break

            chunk_packed, chunk_side, chunk_labels, chunk_lines = pending.popleft().get()
            size = min(len(chunk_labels), max_positions - count)
            packed[count:count + size] = chunk_packed[:size]
            side[count:count + size] = chunk_side[:size]
            labels[count:count + size] = chunk_labels[:size]
            count += size
            num_lines += chunk_lines
            if count >= next_log:
                log(f"Loaded {count:,} / {max_positions:,} positions from {num_lines:,} lines")
                next_log += 5_000_000

    log(f"Done: {count:,} positions from {num_lines:,} lines in {(time.time() - start) / 60:.0f} min")
    return packed[:count], side[:count], labels[:count]


def save_cache(arrays):
    os.makedirs(cache_dir, exist_ok=True)
    for name, array in zip(CACHE_NAMES, arrays):
        np.save(os.path.join(cache_dir, f"{name}.npy"), array)
    with open(os.path.join(cache_dir, "done.json"), "w", encoding="utf-8") as f:
        json.dump({"positions": len(arrays[2]), "min_depth": min_depth}, f)

## Quick check

Reads the first positions of the database. Fails within minutes if the download or the parsing does not work.

In [ ]:
log("Quick check: the first positions of the database...")
check_packed, check_side, check_labels = load_positions(20_000, max_minutes=10)
assert len(check_labels) > 1000, "could not read the Lichess evaluation database"
check_board = np.stack([check_packed & 15, check_packed >> 4], axis=2).reshape(-1, 64)
assert ((check_board == 6).sum(axis=1) == 1).all() and ((check_board == 12).sum(axis=1) == 1).all(), "every position needs exactly one king per side"
log(f"Quick check passed: {len(check_labels):,} positions, white to move in {check_side.mean():.0%}, mean label for white {check_labels.astype(np.float32).mean():+.3f}")

## Stream the whole database and save the cache

In [ ]:
log(f"Streaming the Lichess evaluation database with {num_workers} workers (this takes a few hours)...")
save_cache(load_positions(max_lichess_positions, max_load_minutes))
cache_gb = sum(os.path.getsize(os.path.join(cache_dir, name)) for name in os.listdir(cache_dir)) / 1e9
log(f"Cache saved in {cache_dir}: {sorted(os.listdir(cache_dir))}, {cache_gb:.1f} GB")
with open(os.path.join(cache_dir, "done.json"), encoding="utf-8") as f:
    log(f"done.json: {json.load(f)}")
log("Done. Use this notebook as input of notebook 08b (Add Input -> Your Work -> this notebook)")